# Лабораторна робота № 3
## Діагностика шкал вимірювання

Курс СК26 «Інтелектуальний аналіз та візуалізація даних»

**Тривалість:** 2 год аудиторно + 3 год самостійно · **Оцінка:** 4 бали
**Файли:** `Ковід_дані_навчальні.xlsx`, `ЛР3_помилковий_аналіз.ipynb`

---

Шкала вимірювання визначає, які операції над змінною взагалі мають сенс.
Комп'ютер цього не знає: він порахує середнє від номера телефону й не поскаржиться.
Знати має вимірювач.

У цій роботі ви складаєте **паспорт восьми змінних** свого варіанта, показуєте на
власних даних, які статистики для них недопустимі, і виправляєте чужий аналіз,
де таких помилок десять.

In [1]:
ПРІЗВИЩЕ_ІМЯ = 'Стик Назарій'      # напишіть тут своє прізвище та ім'я
ГРУПА        = 'ШІ-32'
НОМЕР        = 17       # ваш номер за списком групи (1–30)

assert ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР, 'Заповніть три поля вище'
assert 1 <= НОМЕР <= 30, 'Номер має бути від 1 до 30'

In [2]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 60)

ВАРІАНТИ = json.load(open('lr3_variants_student.json', encoding='utf-8'))
В = ВАРІАНТИ[НОМЕР - 1]

print('=' * 74)
print(f'ВАРІАНТ {В["варіант"]}   ·   {ПРІЗВИЩЕ_ІМЯ}   ·   {ГРУПА}')
print('=' * 74)
print()
print('Ваші вісім стовпців:')
for i, c in enumerate(В['стовпці'], 1):
    print(f'   {i}. {c}')
print()
print('Ваша пара для аналізу зв’язку:', ' та '.join(В['пара']))
print()
print('Ваше додаткове питання:')
print('  ', В['питання'])

ВАРІАНТ 17   ·   Стик Назарій   ·   ШІ-32

Ваші вісім стовпців:
   1. ПІП
   2. Ускладнення
   3. Перебіг
   4. Поступлення
   5. Дотація
   6. Т 37-37,5
   7. Сечов2
   8. Гемогл2

Ваша пара для аналізу зв’язку: Стать та помер/жив

Ваше додаткове питання:
   Ви хочете нормалізувати всі числові стовпці формулою (x − min) / (max − min). Знайдіть у файлі стовпець, на якому ця формула зламається, і поясніть чому.


In [3]:
ФАЙЛ = 'Ковід_дані_навчальні.xlsx'
df = pd.read_excel(ФАЙЛ, sheet_name='Лист1', engine='openpyxl')
МОЇ = В['стовпці']

print('Розмір набору:', df.shape)
df[МОЇ].head(8)

Розмір набору: (121, 77)


,ПІП,Ускладнення,Перебіг,Поступлення,Дотація,"Т 37-37,5",Сечов2,Гемогл2
0,Федорюк Петро Юрійович,"тромбоз підколінної артерії, гангрена? ТЕЛА? СЛН ГРДС",важкий,2022-02-24,14 днів,1900-01-11 00:00:00,6.5,144
1,Загул Григорій Сергійович,NaN,важкий,2022-02-22,12 днів,19,4,137
2,Бондара Лариса Павлівна,NaN,важкий,2021-12-30,14 днів,17,10.1,105
3,Ватаманюка Уляна Дмитрівна,NaN,важкий,2021-12-24,16 днів,NaN,12.5,116
4,Бабійа Лариса Олегівна,NaN,важкий,2022-02-12,27,4,7.4,124
5,Ковалика ганна Андріївна,"Респірат дестрес синдром, емфізема легень, СЛН",важкий,2021-12-15,9,9,21.4,120
6,Височан Віктор Володимирович,NaN,важкий,2022-04-23,9,4,6.1,151
7,Дзюбаа Ірина Іванівна,NaN,серед важ,2022-05-18,6,3,7.6,137


---
# Етап 1. Паспорт змінних

Для кожного з восьми стовпців визначте шкалу вимірювання. Спирайтеся не на тип даних
у pandas, а на **зміст величини**: що означають її значення і які порівняння між ними
мають сенс.

Нагадування:

| Шкала | Ознака |
|---|---|
| Номінальна | значення можна лише порівняти на рівність |
| Порядкова | є природний порядок, але відстані між сусідніми рівнями невідомі |
| Інтервальна | різниці мають сенс, нуль умовний |
| Відношень | є природний нуль, відношення мають сенс |
| Абсолютна | природні і нуль, і одиниця вимірювання |

Окремо позначайте **ідентифікатори** — значення, що не є величиною взагалі
(номер історії хвороби, прізвище, телефон).

In [4]:
# Спершу подивіться, що каже pandas — і не довіряйте йому
df[МОЇ].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   ПІП          121 non-null    object        
 1   Ускладнення  61 non-null     object        
 2   Перебіг      121 non-null    object        
 3   Поступлення  120 non-null    datetime64[ns]
 4   Дотація      102 non-null    object        
 5   Т 37-37,5    66 non-null     object        
 6   Сечов2       102 non-null    object        
 7   Гемогл2      105 non-null    object        
dtypes: datetime64[ns](1), object(7)
memory usage: 7.7+ KB


In [5]:
# Далі — що є всередині насправді
for c in МОЇ:
    з = df[c].dropna()
    типи = ', '.join(f'{t.__name__}×{n}' for t, n in з.map(type).value_counts().items())
    print(f'--- {c}')
    print(f'    непорожніх {len(з)} з {len(df)} | унікальних {з.nunique()} | типи: {типи}')
    print(f'    приклади: {[repr(v) for v in з.unique()[:4]]}')
    print()

--- ПІП
    непорожніх 121 з 121 | унікальних 121 | типи: str×121
    приклади: ["'Федорюк Петро Юрійович'", "'Загул Григорій Сергійович'", "'Бондара Лариса Павлівна'", "'Ватаманюка Уляна Дмитрівна'"]

--- Ускладнення
    непорожніх 61 з 121 | унікальних 35 | типи: str×61
    приклади: ["'тромбоз підколінної артерії, гангрена? ТЕЛА? СЛН ГРДС'", "'Респірат дестрес синдром, емфізема легень, СЛН'", "'Респірат дестрес синдром, емфізема легень, СЛН ТІШ'", "'Гострий респіраторний дистрес синдром,ТЕЛА?Прогресуюча серц-леген недос'"]

--- Перебіг
    непорожніх 121 з 121 | унікальних 3 | типи: str×121
    приклади: ["'важкий'", "'серед важ'", "'вкрай важкий '"]

--- Поступлення
    непорожніх 120 з 121 | унікальних 93 | типи: Timestamp×120
    приклади: ["Timestamp('2022-02-24 00:00:00')", "Timestamp('2022-02-22 00:00:00')", "Timestamp('2021-12-30 00:00:00')", "Timestamp('2021-12-24 00:00:00')"]

--- Дотація
    непорожніх 102 з 121 | унікальних 42 | типи: int×96, str×6
    приклади: ["'14 дні

### Заповніть паспорт

Для кожного стовпця вкажіть шкалу та **обґрунтування** — одне речення про те,
чому саме ця шкала. Обґрунтування оцінюється, назва шкали сама по собі — ні.

In [6]:
pd.set_option('display.max_colwidth', None)

ПАСПОРТ = {
    'ПІП':         ('ідентифікатор', 'Унікальне для кожного пацієнта (121 з 121).'),
    'Ускладнення': ('номінальна',    'Назва діагнозу; ускладнення можна просто порівняти на рівність, порядку між ними немає.'),
    'Перебіг':     ('порядкова',     'Рівні «серед важ» < «важкий» < «вкрай важкий» мають порядок, але відстань між ними невідома.'),
    'Поступлення': ('інтервальна',   'Дата, різниця дат у днях має сенс, а нуль календаря умовний, тому відношення дат безглузде.'),
    'Дотація':     ('відношень',     'Кількість днів кисневої дотації: 0 означає її відсутність, 20 днів удвічі довше за 10.'),
    'Т 37-37,5':   ('відношень',     'Кількість днів із субфебрильною температурою: є справжній нуль, відношення мають сенс.'),
    'Сечов2':      ('відношень',     'Концентрація сечовини в ммоль/л: нуль означає відсутність речовини, відношення мають сенс.'),
    'Гемогл2':     ('відношень',     'Концентрація гемоглобіну в г/л: фізичний нуль існує, 140 г/л справді вдвічі більше за 70.'),
}

for c in МОЇ:
    ПАСПОРТ.setdefault(c, ('', ''))

паспорт_df = pd.DataFrame(
    [(c, ПАСПОРТ[c][0], ПАСПОРТ[c][1]) for c in МОЇ],
    columns=['Стовпець', 'Шкала', 'Обґрунтування'])
паспорт_df

,Стовпець,Шкала,Обґрунтування
0,ПІП,ідентифікатор,Унікальне для кожного пацієнта (121 з 121).
1,Ускладнення,номінальна,"Назва діагнозу; ускладнення можна просто порівняти на рівність, порядку між ними немає."
2,Перебіг,порядкова,"Рівні «серед важ» < «важкий» < «вкрай важкий» мають порядок, але відстань між ними невідома."
3,Поступлення,інтервальна,"Дата, різниця дат у днях має сенс, а нуль календаря умовний, тому відношення дат безглузде."
4,Дотація,відношень,"Кількість днів кисневої дотації: 0 означає її відсутність, 20 днів удвічі довше за 10."
5,"Т 37-37,5",відношень,"Кількість днів із субфебрильною температурою: є справжній нуль, відношення мають сенс."
6,Сечов2,відношень,"Концентрація сечовини в ммоль/л: нуль означає відсутність речовини, відношення мають сенс."
7,Гемогл2,відношень,"Концентрація гемоглобіну в г/л: фізичний нуль існує, 140 г/л справді вдвічі більше за 70."


In [7]:
# --- САМОПЕРЕВІРКА 1 ---
ДОПУСТИМІ = {'номінальна', 'порядкова', 'інтервальна', 'відношень',
             'абсолютна', 'ідентифікатор'}
for c in МОЇ:
    ш, об = ПАСПОРТ[c]
    assert ш.strip().lower() in ДОПУСТИМІ, f'{c}: шкалу «{ш}» не розпізнано'
    assert len(об.strip()) >= 25, f'{c}: обґрунтування закоротке'
print('Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.')

Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.


---
# Етап 2. Допустимі статистики

Для кожного стовпця обчисліть **тільки те, що дозволяє його шкала**.

* номінальна та ідентифікатор — мода, частоти
* порядкова — додатково медіана та квантилі
* інтервальна — додатково середнє та стандартне відхилення
* відношень і абсолютна — усе, зокрема коефіцієнт варіації

Якщо для стовпця не можна порахувати нічого, крім частот, — так і напишіть.

In [8]:
import re

def до_числа(v):
    if hasattr(v, 'day'):
        return v.day
    знайдено = re.search(r'\d+\.?\d*', str(v))
    return float(знайдено.group()) if знайдено else np.nan

РІВНІ = ['серед важ', 'важкий', 'вкрай важкий'] # перебіг

def допустима_статистика(стовпець, шкала):
    з = df[стовпець].dropna()
    р = {'непорожніх': len(з), 'унікальних': з.nunique()}
    ш = шкала.strip().lower()
    
    if ш == 'ідентифікатор':
        return р

    # Крок 1. Приводимо значення до потрібного вигляду
    if ш == 'відношень':
        з = з.map(до_числа).dropna()
    elif ш == 'інтервальна':
        з = pd.to_datetime(з)
    else:
        з = з.astype(str).str.strip()

    # Крок 2. Мода й частоти для всіх шкал
    р['мода'] = з.mode()[0]
    р['топ-3 частоти'] = з.value_counts().head(3).to_dict()

    # Крок 3. Порядкова, медіана й квантилі через номери рівнів
    if ш == 'порядкова':
        номери = з.map(РІВНІ.index)
        for назва, q in [('Q1', 0.25), ('медіана', 0.5), ('Q3', 0.75)]:
            р[назва] = РІВНІ[int(номери.quantile(q))]

    # середнє та ст. відхилення
    if ш in ('інтервальна', 'відношень'):
        р['медіана'] = з.median()
        р['середнє'] = з.mean()
        р['ст. відхилення'] = з.std()

    # коефіцієнт варіації
    if ш == 'відношень':
        р['CV, %'] = round(100 * з.std() / з.mean(), 1)

    return р


for c in МОЇ:
    print(f'--- {c}  ({ПАСПОРТ[c][0]})')
    print('   ', допустима_статистика(c, ПАСПОРТ[c][0]))
    print()

--- ПІП  (ідентифікатор)
    {'непорожніх': 121, 'унікальних': 121}

--- Ускладнення  (номінальна)
    {'непорожніх': 61, 'унікальних': 35, 'мода': 'Гострий респіраторний дистрес синдром. Гостра серц-леген недос .Гостра емфізема легень.Токсико-інфекційний шок', 'топ-3 частоти': {'Гострий респіраторний дистрес синдром. Гостра серц-леген недос .Гостра емфізема легень.Токсико-інфекційний шок': 14, 'ГРДС': 5, 'Прогресуюча серц-леген недос .Набряк легень . Сладж синдром': 3}}

--- Перебіг  (порядкова)
    {'непорожніх': 121, 'унікальних': 3, 'мода': 'важкий', 'топ-3 частоти': {'важкий': 90, 'серед важ': 30, 'вкрай важкий': 1}, 'Q1': 'важкий', 'медіана': 'важкий', 'Q3': 'важкий'}

--- Поступлення  (інтервальна)
    {'непорожніх': 120, 'унікальних': 93, 'мода': Timestamp('2022-04-02 00:00:00'), 'топ-3 частоти': {Timestamp('2022-04-02 00:00:00'): 4, Timestamp('2022-04-05 00:00:00'): 4, Timestamp('2021-09-01 00:00:00'): 3}, 'медіана': Timestamp('2022-02-27 12:00:00'), 'середнє': Timestamp('2022

---
# Етап 3. Три безглузді числа

`df.describe()` порахує статистики для всього, що схоже на число. Частина цих чисел
не має жодного змісту.

Знайдіть **щонайменше три** такі числа у своєму наборі: обчисліть їх, а потім
поясніть, чому кожне з них безглузде.

In [9]:
# Очищена копія
чисті = df[МОЇ].copy()
for c in ['Дотація', 'Т 37-37,5', 'Сечов2', 'Гемогл2']:
    чисті[c] = чисті[c].dropna().map(до_числа)
чисті['Перебіг'] = чисті['Перебіг'].str.strip()

чисті.describe(include='all').T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
ПІП,121,121,Федорюк Петро Юрійович,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ускладнення,61,35,Гострий респіраторний дистрес синдром. Гостра серц-леген недос .Гостра емфізема легень.Токсико-інфекційний шок,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Перебіг,121,3,важкий,90,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Поступлення,120,NaN,NaN,NaN,2022-01-12 07:24:00,2021-02-25 00:00:00,2021-11-05 06:00:00,2022-02-27 12:00:00,2022-04-07 00:00:00,2022-06-03 00:00:00,NaN
Дотація,101.0,NaN,NaN,NaN,16.732673,1.0,8.0,12.0,22.0,79.0,13.759281
"Т 37-37,5",65.0,NaN,NaN,NaN,4.692308,1.0,2.0,3.0,5.0,19.0,3.912812
Сечов2,101.0,NaN,NaN,NaN,10.198614,3.4,5.9,7.3,10.0,116.0,13.274622
Гемогл2,104.0,NaN,NaN,NaN,128.711538,51.0,116.0,134.0,144.0,190.0,23.514372


In [10]:
pd.set_option('display.max_colwidth', None)

БЕЗГЛУЗДІ = [
    ('ПІП', 'top / freq', 'Федорюк Петро Юрійович, 1',
     'Усі 121 ім’я різні, «мода» — випадково перше ім’я.'),
    ('Ускладнення', 'top / freq', 'Гострий респіраторний дистрес синдром…, 14',
     'Те саме записано і як «ГРДС» (ще 5 разів), тож частота занижена.'),
    ('Поступлення', '50%', '2022-02-27 12:00:00',
     'Час 12:00 вигаданий: у даних є лише дати, pandas взяв середину між двома днями.'),
]


assert len(БЕЗГЛУЗДІ) >= 3, 'Потрібно щонайменше три приклади'
pd.DataFrame(БЕЗГЛУЗДІ, columns=['Стовпець', 'Статистика', 'Значення', 'Чому безглузде'])

,Стовпець,Статистика,Значення,Чому безглузде
0,ПІП,top / freq,"Федорюк Петро Юрійович, 1","Усі 121 ім’я різні, «мода» — випадково перше ім’я."
1,Ускладнення,top / freq,"Гострий респіраторний дистрес синдром…, 14","Те саме записано і як «ГРДС» (ще 5 разів), тож частота занижена."
2,Поступлення,50%,2022-02-27 12:00:00,"Час 12:00 вигаданий: у даних є лише дати, pandas взяв середину між двома днями."


---
# Етап 4. Міра зв'язку для вашої пари

Для двох змінних вашого варіанта треба оцінити зв'язок. Міру обирає **пара шкал**,
а не зручність:

| Поєднання шкал | Міра |
|---|---|
| відношень × відношень | кореляція Пірсона |
| порядкова × порядкова | рангова кореляція Спірмена |
| номінальна × номінальна | χ² та коефіцієнт Крамера (V) |
| відношень × бінарна номінальна | точковий бісеріальний коефіцієнт |
| номінальна × порядкова | коефіцієнт Крамера (V) |

Обґрунтуйте вибір, обчисліть міру й поясніть результат. Окремо покажіть, **що
станеться**, якщо застосувати Пірсона там, де він недопустимий, — і чому отримане
число не можна інтерпретувати.

In [11]:
A, B = В['пара']
print('Ваша пара:', A, '×', B)
print()
print(f'{A}: шкала -', ПАСПОРТ.get(A, ('не у вашому списку',))[0] or '(визначте самостійно)')
print(f'{B}: шкала -', ПАСПОРТ.get(B, ('не у вашому списку',))[0] or '(визначте самостійно)')
print()
print(df[[A, B]].head(6).to_string())

Ваша пара: Стать × помер/жив

Стать: шкала - не у вашому списку
помер/жив: шкала - не у вашому списку

  Стать  помер/жив
0     ч          1
1     ч          2
2     ж          2
3     ж          2
4     ж          2
5     ж          1


In [12]:
from scipy.stats import chi2_contingency

ОБРАНА_МІРА = 'χ² та коефіцієнт Крамера V'
ЧОМУ_САМЕ_ВОНА = ('Обидві змінні номінальні: стать і результат лікування - це мітки без порядку '
                  'та відстаней, тому можна лише рахувати частоти комбінацій.')

assert ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40, 'Заповніть обидва поля'

# Чистимо стать
стать = df['Стать'].str.strip().str[0].map({'ж': 'жінка', 'ч': 'чоловік'})
результат = df['помер/жив'].map({1: 'помер', 2: 'вижив'})

# Таблиця спряженості
таблиця = pd.crosstab(стать, результат)
print(таблиця, '\n')
print('Частка в кожній групі:')
print(pd.crosstab(стать, результат, normalize='index').round(2), '\n')

# χ² і V Крамера
chi2, p, _, _ = chi2_contingency(таблиця, correction=False)
n = таблиця.values.sum()
V = np.sqrt(chi2 / (n * (min(таблиця.shape) - 1)))
print(f'χ² = {chi2:.2f},  p = {p:.3f},  V Крамера = {V:.3f}')

помер/жив  вижив  помер
Стать                  
жінка         32     43
чоловік       28     18 

Частка в кожній групі:
помер/жив  вижив  помер
Стать                  
жінка       0.43   0.57
чоловік     0.61   0.39 

χ² = 3.78,  p = 0.052,  V Крамера = 0.177


χ² = 3.78 -> наскільки таблиця відрізняється від «зв'язку немає»  
p = 0.052 -> ймовірність побачити таку різницю випадково; > 0.05, тому не доведено  
V Крамера =0.177 -> сила зв'язку від 0 до 1; 0.1–0.3 - слабкий

In [13]:
# Сирі дані
коди_сирі = df['Стать'].astype('category').cat.codes
print('Сирі коди:', dict(enumerate(df['Стать'].astype('category').cat.categories)))
print(f'Пірсон r = {коди_сирі.corr(df["помер/жив"]):.3f}')
print('-> 4 «статі» і довільні номери: «чол»=3 нібито «втричі більше», ніж «жін»=1.')
print()

# Чистка за статтю
r1 = стать.map({'жінка': 0, 'чоловік': 1}).corr(df['помер/жив'])
r2 = стать.map({'жінка': 1, 'чоловік': 0}).corr(df['помер/жив'])
print(f'жінка=0, чоловік=1:  r = {r1:+.3f}')
print(f'жінка=1, чоловік=0:  r = {r2:+.3f}')
print('-> Ті самі люди, а знак протилежний. У статі немає напрямку «більше/менше»,')
print('   тому знак і саме число Пірсона тут нічого не означають.')
print('   Модуль 0.177 збігся з V Крамера лише тому, що обидві змінні мають по 2 категорії.')

Сирі коди: {0: 'ж', 1: 'жін', 2: 'ч', 3: 'чол'}
Пірсон r = 0.248
-> 4 «статі» і довільні номери: «чол»=3 нібито «втричі більше», ніж «жін»=1.

жінка=0, чоловік=1:  r = +0.177
жінка=1, чоловік=0:  r = -0.177
-> Ті самі люди, а знак протилежний. У статі немає напрямку «більше/менше»,
   тому знак і саме число Пірсона тут нічого не означають.
   Модуль 0.177 збігся з V Крамера лише тому, що обидві змінні мають по 2 категорії.


---
# Етап 5. Підготовка змінних до моделі

Шкала визначає не лише статистику, а й спосіб кодування.

| Шкала | Як готують до моделі |
|---|---|
| Номінальна | one-hot кодування; порядок категорій не вводять |
| Порядкова | впорядкована категорія або цілі коди із **заданим** порядком |
| Інтервальна | стандартизація; для дат — перехід до тривалостей |
| Відношень | нормалізація або стандартизація; можливе логарифмування |
| Ідентифікатор | у модель не беруть узагалі |

Для кожного свого стовпця вкажіть спосіб підготовки — і **виконайте його**
щонайменше для трьох стовпців різних шкал.

In [14]:
pd.set_option('display.max_colwidth', None)

ПІДГОТОВКА = {
    'ПІП':         'Викинути: ідентифікатор, нічого не вимірює.',
    'Ускладнення': 'Викинути: заповнене лише в померлих (61 з 61) - витік цілі; інакше one-hot.',
    'Перебіг':     'Цілі коди в заданому порядку: серед важ=0, важкий=1, вкрай важкий=2.',
    'Поступлення': 'Дата -> кількість днів від першої госпіталізації, потім стандартизація.',
    'Дотація':     'Очистити до числа, log1p через довгий хвіст (до 79 днів), нормалізація.',
    'Т 37-37,5':   'Очистити до числа (тексти «2 дні в стац»), нормалізація min-max.',
    'Сечов2':      'Очистити до числа, log1p через викид 116, потім стандартизація.',
    'Гемогл2':     'Очистити до числа, нормалізація min-max до діапазону [0, 1].',
}
for c in МОЇ:
    ПІДГОТОВКА.setdefault(c, '')

pd.DataFrame([(c, ПАСПОРТ[c][0], ПІДГОТОВКА[c]) for c in МОЇ],
             columns=['Стовпець', 'Шкала', 'Як готуємо'])

,Стовпець,Шкала,Як готуємо
0,ПІП,ідентифікатор,"Викинути: ідентифікатор, нічого не вимірює."
1,Ускладнення,номінальна,Викинути: заповнене лише в померлих (61 з 61) - витік цілі; інакше one-hot.
2,Перебіг,порядкова,"Цілі коди в заданому порядку: серед важ=0, важкий=1, вкрай важкий=2."
3,Поступлення,інтервальна,"Дата -> кількість днів від першої госпіталізації, потім стандартизація."
4,Дотація,відношень,"Очистити до числа, log1p через довгий хвіст (до 79 днів), нормалізація."
5,"Т 37-37,5",відношень,"Очистити до числа (тексти «2 дні в стац»), нормалізація min-max."
6,Сечов2,відношень,"Очистити до числа, log1p через викид 116, потім стандартизація."
7,Гемогл2,відношень,"Очистити до числа, нормалізація min-max до діапазону [0, 1]."


In [ ]:
# ваш код: реалізуйте підготовку для трьох стовпців різних шкал

# 1) Перебіг: слова -> коди в нашому порядку
перебіг_код = df['Перебіг'].str.strip().map(РІВНІ.index)

# 2) Поступлення: дата -> дні від першої госпіталізації -> стандартизація
дати = pd.to_datetime(df['Поступлення'])
дні = (дати - дати.min()).dt.days
поступлення_ст = (дні - дні.mean()) / дні.std()

# 3) Гемогл2: очистити -> нормалізація min-max
гемогл = df['Гемогл2'].map(до_числа)
гемогл_норм = (гемогл - гемогл.min()) / (гемогл.max() - гемогл.min())

# Перевірка витоку для Ускладнення
print('Ускладнення заповнене × результат:')
print(pd.crosstab(df['Ускладнення'].notna(), df['помер/жив']), '\n')

готові = pd.DataFrame({
    'Перебіг (код)':        перебіг_код,
    'Поступлення (станд.)': поступлення_ст.round(2),
    'Гемогл2 (норм.)':      гемогл_норм.round(2),
})
print(готові.head(8), '\n')
print(готові.describe().round(2))


Ускладнення заповнене × результат:
помер/жив     1   2
Ускладнення        
False         0  60
True         61   0 

   Перебіг (код)  Поступлення (станд.)  Гемогл2 (норм.)
0              1                  0.38             0.67
1              1                  0.36             0.62
2              1                 -0.12             0.39
3              1                 -0.17             0.47
4              1                  0.27             0.53
5              1                 -0.25             0.50
6              1                  0.89             0.72
7              0                  1.11             0.62 

       Перебіг (код)  Поступлення (станд.)  Гемогл2 (норм.)
count         121.00                120.00           104.00
mean            0.76                 -0.00             0.56
std             0.45                  1.00             0.17
min             0.00                 -2.84             0.00
25%             1.00                 -0.60             0.47
50%             1

**Перед нормалізацією** обов'язково перевірте варіативність. Стовпець, у якому
всі значення однакові, дасть нуль у знаменнику — і мовчазний `NaN` замість помилки.

In [16]:
for c in МОЇ:
    з = pd.to_numeric(df[c], errors='coerce').dropna()
    if len(з) and з.nunique() == 1:
        print(f'УВАГА: {c} — сталий стовпець, нормалізація дасть ділення на нуль')
    elif len(з) and з.max() == з.min():
        print(f'УВАГА: {c} — max дорівнює min')
print('Перевірку варіативності виконано.')

Перевірку варіативності виконано.


In [17]:
# --- САМОПЕРЕВІРКА 2 ---
assert all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ), \
    'Для кожного стовпця вкажіть спосіб підготовки'
assert len(БЕЗГЛУЗДІ) >= 3, 'Етап 3 не завершено'
print('Самоперевірка 2 пройдена.')

Самоперевірка 2 пройдена.


---
# Етап 6. Виправлення чужого аналізу

Відкрийте файл **`ЛР3_помилковий_аналіз.ipynb`**. Це звіт, який виконується
без жодної помилки й містить десять хибних місць.

Для кожного з десяти заповніть рядок нижче:

* **розділ** — номер розділу звіту;
* **що не так** — яка шкала порушена або яке припущення хибне;
* **чому число неправильне** — що саме воно означає насправді;
* **як правильно** — коректний варіант або обґрунтована відмова від показника.

In [18]:
pd.set_option('display.max_colwidth', None)

ВИПРАВЛЕННЯ = [
    (1, 'Середнє для номінальної шкали (помер/жив)',
     '1 і 2 - лише мітки; «1.50» не означає «напівживий», а лише що померлих і живих порівну.',
     'Частоти: померли 61 (50.4%), вижили 60 (49.6%).'),
    (2, 'Брудні дані: одна стать записана по-різному',
     '«ж» і «жін», «ч» і «чол» - те саме; «4 статі» - помилка запису, а не факт.',
     'Звести до 2 категорій (перша літера): жінок 75, чоловіків 46.'),
    (3, 'Порядкова шкала: алфавітний порядок і середнє кодів',
     'sort_index сортує за алфавітом (найлегший - «важкий»), дублікати через пробіли й описки; середнє рангів недопустиме.',
     'Очистити до 5 рівнів, задати порядок легкий<…<критичний, рахувати медіану: «середній».'),
    (4, 'Число з непридатних даних: КТ у двох форматах',
     'to_numeric викинув 17 записів на кшталт «70-80%»; середнє 39.6% пораховане лише по 27 з 44.',
     'Перевести все у відсотки (0.3→30, «70-80%»→75): 43 значення, середнє 42.5%, медіана 35%.'),
    (5, 'Два несумісні кодування сатурації',
     'Одне значення записане як частка 0.9 замість 90%; воно занижує середнє і дає абсурдний мінімум 0.9%.',
     'Значення ≤1 помножити на 100: середнє 82.8%, медіана 84%, мінімум 50%.'),
    (6, 'Коефіцієнт варіації для інтервальної шкали (дати)',
     'Нуль дати умовний (1970 р.), тож CV залежить від точки відліку; 463 дні розкиду виглядають як 0.6%.',
     'Для дат CV не рахують: подати розмах (463 дні) і ст. відхилення (113 днів).'),
    (7, 'Пірсон для двох номінальних змінних + хибний висновок',
     'Стать і результат - мітки, тож потрібні χ² і V; r=0.177 не «близький до нуля», а слабкий зв’язок на межі значущості.',
     'χ² = 3.78, p = 0.052, V Крамера = 0.18: летальність жінок 57% проти 39%, зв’язок слабкий і не доведений.'),
    (8, 'Нормалізація всього підряд і сталий стовпець',
     'Нормалізовано номінальний «помер/жив»; ФЕРИТ2 має одне значення, max = min, 0/0 дає мовчазний NaN.',
     'Перед нормалізацією перевірити шкалу та варіативність; ФЕРИТ2 і номінальні стовпці виключити.'),
    (9, 'dropna() видалив усі рядки',
     'У кожному рядку є хоч один пропуск (ФЕРИТ2 порожній у 120 з 121), тому лишилось 0 рядків — набір порожній.',
     'Видаляти не рядки, а майже порожні стовпці; решту пропусків заповнювати або аналізувати окремо.'),
    (10, 'Витік цілі: дата розтину',
     'Розтин роблять лише померлим (61 з 61), тож ця ознака - сама відповідь, а не предиктор; r = 1.00 це й показує.',
     'Виключити з моделі всі ознаки, відомі лише після результату: «Дата розтину», «Ускладнення».'),
]

assert len(ВИПРАВЛЕННЯ) == 10, f'Знайдено {len(ВИПРАВЛЕННЯ)} помилок із 10'
for р in ВИПРАВЛЕННЯ:
    assert len(р) == 4 and all(str(x).strip() for x in р), 'Заповніть усі чотири поля'
    assert len(str(р[2])) >= 30 and len(str(р[3])) >= 25, 'Пояснення закороткі'

pd.DataFrame(ВИПРАВЛЕННЯ, columns=['Розділ', 'Що не так', 'Чому число неправильне', 'Як правильно'])

,Розділ,Що не так,Чому число неправильне,Як правильно
0,1,Середнє для номінальної шкали (помер/жив),"1 і 2 - лише мітки; «1.50» не означає «напівживий», а лише що померлих і живих порівну.","Частоти: померли 61 (50.4%), вижили 60 (49.6%)."
1,2,Брудні дані: одна стать записана по-різному,"«ж» і «жін», «ч» і «чол» - те саме; «4 статі» - помилка запису, а не факт.","Звести до 2 категорій (перша літера): жінок 75, чоловіків 46."
2,3,Порядкова шкала: алфавітний порядок і середнє кодів,"sort_index сортує за алфавітом (найлегший - «важкий»), дублікати через пробіли й описки; середнє рангів недопустиме.","Очистити до 5 рівнів, задати порядок легкий<…<критичний, рахувати медіану: «середній»."
3,4,Число з непридатних даних: КТ у двох форматах,to_numeric викинув 17 записів на кшталт «70-80%»; середнє 39.6% пораховане лише по 27 з 44.,"Перевести все у відсотки (0.3→30, «70-80%»→75): 43 значення, середнє 42.5%, медіана 35%."
4,5,Два несумісні кодування сатурації,Одне значення записане як частка 0.9 замість 90%; воно занижує середнє і дає абсурдний мінімум 0.9%.,"Значення ≤1 помножити на 100: середнє 82.8%, медіана 84%, мінімум 50%."
5,6,Коефіцієнт варіації для інтервальної шкали (дати),"Нуль дати умовний (1970 р.), тож CV залежить від точки відліку; 463 дні розкиду виглядають як 0.6%.",Для дат CV не рахують: подати розмах (463 дні) і ст. відхилення (113 днів).
6,7,Пірсон для двох номінальних змінних + хибний висновок,"Стать і результат - мітки, тож потрібні χ² і V; r=0.177 не «близький до нуля», а слабкий зв’язок на межі значущості.","χ² = 3.78, p = 0.052, V Крамера = 0.18: летальність жінок 57% проти 39%, зв’язок слабкий і не доведений."
7,8,Нормалізація всього підряд і сталий стовпець,"Нормалізовано номінальний «помер/жив»; ФЕРИТ2 має одне значення, max = min, 0/0 дає мовчазний NaN.",Перед нормалізацією перевірити шкалу та варіативність; ФЕРИТ2 і номінальні стовпці виключити.
8,9,dropna() видалив усі рядки,"У кожному рядку є хоч один пропуск (ФЕРИТ2 порожній у 120 з 121), тому лишилось 0 рядків — набір порожній.","Видаляти не рядки, а майже порожні стовпці; решту пропусків заповнювати або аналізувати окремо."
9,10,Витік цілі: дата розтину,"Розтин роблять лише померлим (61 з 61), тож ця ознака - сама відповідь, а не предиктор; r = 1.00 це й показує.","Виключити з моделі всі ознаки, відомі лише після результату: «Дата розтину», «Ускладнення»."


### Виправте щонайменше три помилки в коді

Оберіть три з десяти та покажіть правильне обчислення поруч із неправильним.

In [19]:
# ваш код: три правильні обчислення замість хибних

# Виправлення 1. Розділ 3: ступінь КТ (порядкова шкала)
РІВНІ_КТ = ['легкий', 'середній', 'важкий', 'вкрай важкий', 'критичний']
перші3 = {'лег': 'легкий', 'сер': 'середній', 'важ': 'важкий', 'вкр': 'вкрай важкий', 'кри': 'критичний'}
ступінь = df['ступінь КТ'].str.strip().str[:3].map(перші3)
print('РОЗДІЛ 3. Ступінь КТ')
print('  було: 12 «рівнів» за алфавітом, «середнє» 5.9 -> легкий')
print('  стало:', ступінь.value_counts().reindex(РІВНІ_КТ).to_dict())
print('  медіана:', РІВНІ_КТ[int(ступінь.dropna().map(РІВНІ_КТ.index).median())])
print()

# Виправлення 2. Розділ 4: КТ у відсотках (два записи + діапазони)
def кт_у_відсотках(v):
    if isinstance(v, (int, float)):
        return v * 100 if v <= 1 else v
    числа = [float(x) for x in re.findall(r'\d+', str(v))]
    return np.mean(числа) if числа else np.nan
кт_було = pd.to_numeric(df['КТ'], errors='coerce') * 100
кт_стало = df['КТ'].dropna().map(кт_у_відсотках).dropna()
print('РОЗДІЛ 4. Обсяг ураження легень, %')
print(f'  було: {кт_було.count()} значень, середнє {кт_було.mean():.1f}, медіана {кт_було.median():.1f}')
print(f'  стало: {кт_стало.count()} значень, середнє {кт_стало.mean():.1f}, медіана {кт_стало.median():.1f}')
print()

# Виправлення 3. Розділ 5: сатурація (частка 0.9 замість 90 %)
s = df['spO2 без дот']
s_стало = s.where(s > 1, s * 100)
print('РОЗДІЛ 5. Сатурація без дотації, %')
print(f'  було: середнє {s.mean():.2f}, мінімум {s.min()}')
print(f'  стало: середнє {s_стало.mean():.2f}, медіана {s_стало.median()}, мінімум {s_стало.min()}')

РОЗДІЛ 3. Ступінь КТ
  було: 12 «рівнів» за алфавітом, «середнє» 5.9 -> легкий
  стало: {'легкий': 7, 'середній': 18, 'важкий': 10, 'вкрай важкий': 4, 'критичний': 3}
  медіана: середній

РОЗДІЛ 4. Обсяг ураження легень, %
  було: 27 значень, середнє 39.6, медіана 30.0
  стало: 43 значень, середнє 42.5, медіана 35.0

РОЗДІЛ 5. Сатурація без дотації, %
  було: середнє 82.05, мінімум 0.9
  стало: середнє 82.79, медіана 84.0, мінімум 50.0


---
# Етап 7. Висновки та індивідуальне питання

In [20]:
ВИСНОВКИ = [
    'Стовпець «Ускладнення» заповнений рівно в усіх 61 померлого і порожній в усіх 60 '
    'тих, хто вижив, тому для прогнозу летальності це витік цілі, і в модель його брати не можна.',

    'Летальність серед жінок вища, ніж серед чоловіків (57% проти 39%), але зв’язок статі з '
    'результатом слабкий (V Крамера = 0.18) і статистично не доведений (χ² = 3.78, p = 0.052).',

    'Числові стовпці містять текст і викиди: у «Т 37-37,5» 38 з 66 значень записані словами, '
    'а викид 116 у «Сечов2» піднімає середнє до 10.2 при медіані 7.3, тож медіана тут надійніша.',
]
assert len(ВИСНОВКИ) >= 3 and all(len(в) >= 60 for в in ВИСНОВКИ), \
    'Потрібні три змістовні висновки'
for i, в in enumerate(ВИСНОВКИ, 1):
    print(f'{i}. {в}')

1. Стовпець «Ускладнення» заповнений рівно в усіх 61 померлого і порожній в усіх 60 тих, хто вижив, тому для прогнозу летальності це витік цілі, і в модель його брати не можна.
2. Летальність серед жінок вища, ніж серед чоловіків (57% проти 39%), але зв’язок статі з результатом слабкий (V Крамера = 0.18) і статистично не доведений (χ² = 3.78, p = 0.052).
3. Числові стовпці містять текст і викиди: у «Т 37-37,5» 38 з 66 значень записані словами, а викид 116 у «Сечов2» піднімає середнє до 10.2 при медіані 7.3, тож медіана тут надійніша.


In [21]:
print('ВАШЕ ПИТАННЯ:'); print(В['питання']); print()
ВІДПОВІДЬ = (
    'Формула ламається на стовпці «ФЕРИТ2»: із 121 пацієнта він заповнений лише в одного '
    '(значення 1265), тому max = min = 1265 і знаменник max − min дорівнює нулю. '
    'Ділення 0/0 не визначене, але pandas не видає помилку, а мовчки записує NaN у весь стовпець. '
    'Саме так сталося у звіті колеги (розділ 8): ФЕРИТ2 після нормалізації має count = 0, '
    'хоча у висновку написано, що всі ознаки приведено до [0, 1]. '
    'Правильно - перед нормалізацією перевіряти варіативність (nunique() > 1) і такі стовпці '
    'виключати: одне значення не несе жодної інформації для моделі. '
    'Крім того, формулу не можна застосовувати до номінальних стовпців на кшталт «помер/жив», '
    'а викиди (як 116 у «Сечов2») стискають усі інші значення біля нуля.'
)
assert len(ВІДПОВІДЬ.strip()) >= 200, 'Відповідь має бути розгорнутою (від 200 символів)'
print(ВІДПОВІДЬ)

ВАШЕ ПИТАННЯ:
Ви хочете нормалізувати всі числові стовпці формулою (x − min) / (max − min). Знайдіть у файлі стовпець, на якому ця формула зламається, і поясніть чому.

Формула ламається на стовпці «ФЕРИТ2»: із 121 пацієнта він заповнений лише в одного (значення 1265), тому max = min = 1265 і знаменник max − min дорівнює нулю. Ділення 0/0 не визначене, але pandas не видає помилку, а мовчки записує NaN у весь стовпець. Саме так сталося у звіті колеги (розділ 8): ФЕРИТ2 після нормалізації має count = 0, хоча у висновку написано, що всі ознаки приведено до [0, 1]. Правильно - перед нормалізацією перевіряти варіативність (nunique() > 1) і такі стовпці виключати: одне значення не несе жодної інформації для моделі. Крім того, формулу не можна застосовувати до номінальних стовпців на кшталт «помер/жив», а викиди (як 116 у «Сечов2») стискають усі інші значення біля нуля.


---
# Етап 8. Декларація про використання інструментів ШІ

Користуватися мовними моделями дозволено. Це нормальна інженерна практика.

**Обов'язковою є прозорість:** ви вказуєте, де саме і як скористалися інструментом —
так само, як у науковій статті вказують використане програмне забезпечення.
Задекларована допомога порушенням не є.

Порушенням є інше: здати роботу, яку ви не можете пояснити. На захисті вас
попросять показати конкретний рядок свого коду, пояснити, чому він саме такий,
і внести в нього невелику зміну на місці.

Для кожного етапу оберіть одне з формулювань:

* `'ні'` — інструментами ШІ не користувався
* `'синтаксис'` — питав про назву функції або синтаксис, код писав сам
* `'код, перевірено'` — код згенеровано, я його прочитав, перевірив і розумію
* `'текст, перероблено'` — згенеровано чернетку тексту, яку я переписав по суті
* `'текст, як є'` — згенерований текст залишено майже без змін

In [22]:
ДЕКЛАРАЦІЯ = {
    'етап 1 — паспорт змінних':        'текст, перероблено',
    'етап 2 — допустимі статистики':   'код, перевірено',
    'етап 3 — безглузді числа':        'код, перевірено',
    'етап 4 — міра зв’язку':           'код, перевірено',
    'етап 5 — підготовка до моделі':   'код, перевірено',
    'етап 6 — виправлення аналізу':    'код, перевірено',
    'етап 7 — висновки й відповідь':   'текст, як є',
}

ЩО_САМЕ_Я_ПЕРЕВІРИВ = (
    'Запускав кожну комірку й звіряв вивід із поясненнями; просив розібрати код по рядках '
    '(функція до_числа, квантилі, χ² і V Крамера, нормалізація); сам перевірив таблицю describe, '
    'частоти статі й результату, витік у стовпці «Ускладнення» та нуль у знаменнику для ФЕРИТ2.'
)
ПІДТВЕРДЖУЮ = True     # я можу пояснити кожен рядок свого коду
                       # і внести в нього зміни на вимогу під час захисту

In [23]:
# --- ФІНАЛЬНА САМОПЕРЕВІРКА ---
ФОРМУЛЮВАННЯ = {'ні', 'синтаксис', 'код, перевірено', 'текст, перероблено', 'текст, як є'}

перевірки = {
    'заповнено титульні поля':        bool(ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР),
    'етап 1: паспорт із 8 стовпців':  all(ПАСПОРТ[c][0] and ПАСПОРТ[c][1] for c in МОЇ),
    'етап 3: три безглузді числа':    len(БЕЗГЛУЗДІ) >= 3,
    'етап 4: міру обрано й пояснено': bool(ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40),
    'етап 5: підготовку описано':     all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ),
    'етап 6: знайдено 10 помилок':    len(ВИПРАВЛЕННЯ) == 10,
    'етап 7: три висновки':           len(ВИСНОВКИ) >= 3,
    'етап 7: відповідь на питання':   len(ВІДПОВІДЬ.strip()) >= 200,
    'декларацію заповнено':           all(v in ФОРМУЛЮВАННЯ for v in ДЕКЛАРАЦІЯ.values()),
    'опис перевірки ≥ 120 символів':  len(ЩО_САМЕ_Я_ПЕРЕВІРИВ.strip()) >= 120,
    'підтвердження авторства':        ПІДТВЕРДЖУЮ is True,
}

print('ЧЕК-ЛИСТ ГОТОВНОСТІ')
print('=' * 52)
for назва, ок in перевірки.items():
    print(('  ГОТОВО     ' if ок else '  НЕ ГОТОВО ') + назва)
print('=' * 52)

assert all(перевірки.values()), 'Робота не готова до здачі — див. чек-лист вище'
print()
print(f'Робота варіанта {В["варіант"]} готова до захисту.')
print(f'{ПРІЗВИЩЕ_ІМЯ}, {ГРУПА}')

ЧЕК-ЛИСТ ГОТОВНОСТІ
  ГОТОВО     заповнено титульні поля
  ГОТОВО     етап 1: паспорт із 8 стовпців
  ГОТОВО     етап 3: три безглузді числа
  ГОТОВО     етап 4: міру обрано й пояснено
  ГОТОВО     етап 5: підготовку описано
  ГОТОВО     етап 6: знайдено 10 помилок
  ГОТОВО     етап 7: три висновки
  ГОТОВО     етап 7: відповідь на питання
  ГОТОВО     декларацію заповнено
  ГОТОВО     опис перевірки ≥ 120 символів
  ГОТОВО     підтвердження авторства

Робота варіанта 17 готова до захисту.
Стик Назарій, ШІ-32


---
### Перед здачею

Виконайте **Kernel → Restart & Run All**. Ноутбук має пройти згори донизу
без жодної помилки — це перше, що перевіряється на захисті.